# ASAP8 manual longitudinal ROI identity registration

Use this notebook to **manually assign the same biological neuron across sessions** for one mouse at a time.

The notebook loads every session/DMD reference image and every extracted ROI mask, then lets you assign a persistent `global_cell_id` (for example `852835_C003`) across days.

## Important behavior

### The CSV always contains *all* anatomical ROI rows
The subject-level `roi_identity_registration.csv` is built from the full per-session ROI manifest. Even if a mask is:
- unassigned,
- QC-failing, or
- hidden by the optional display filter,

its row still remains in the saved CSV. This is important for downstream cross-session reporting.

### QC visibility is a toggle
Use:

```python
APPLY_ROI_QC_FILTER = False
```

to show **all ROIs** in the GUI. This is the default and is what you want if you intend to click through every extracted ROI.

Set it to `True` only if you want the GUI to **show/click only QC-passing ROIs**. This affects the viewer, ROI dropdown, and click hit-testing — **not** whether the ROI exists in the saved registration table.

### Manual exclusion is separate
`excluded=True` is a manual annotation, not an automated QC decision.


In [ ]:

%load_ext autoreload
%autoreload 2

from pathlib import Path
from datetime import datetime
import hashlib
import json
import re
import shutil
import warnings

import h5py
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import cm
from IPython.display import display, HTML, clear_output

try:
    import ipywidgets as widgets
except ImportError as exc:
    raise ImportError("This notebook requires ipywidgets. Install with: pip install ipywidgets") from exc

# Direct image clicking requires the ipympl/widget backend. The registrar remains fully
# usable through the explicit ROI dropdown + ASSIGN button if this is unavailable.
CLICK_BACKEND = False
CLICK_BACKEND_ERROR = None
try:
    import ipympl  # noqa: F401
    get_ipython().run_line_magic("matplotlib", "widget")
    CLICK_BACKEND = True
except Exception as exc:
    CLICK_BACKEND_ERROR = repr(exc)
    get_ipython().run_line_magic("matplotlib", "inline")

from vip_slap2_analysis.io.session_registry import VIPSessionRegistry

display(HTML("<style>.container { width:100% !important; }</style>"))

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

print("Matplotlib backend:", matplotlib.get_backend())
print("Direct ROI clicking:", "ENABLED" if CLICK_BACKEND else "DISABLED")
if not CLICK_BACKEND:
    print("\nDirect clicking could not be enabled.")
    print("The GUI still works with the ROI dropdown + ASSIGN button.")
    print("To enable clicking, install ipympl in this kernel and restart it:")
    print("    %pip install ipympl")
    print("Then restart the kernel and rerun the notebook from the top.")
    if CLICK_BACKEND_ERROR:
        print("Backend error:", CLICK_BACKEND_ERROR)



## 0. Click-backend test

**Do this before loading the mouse.** If direct clicking is enabled, click anywhere in the test image below.
The text immediately under the image must change to `CLICK RECEIVED ...`.

If it does not change, do not troubleshoot the biological data—the Jupyter matplotlib backend is not delivering click events. You can either:

- install/restart with `ipympl`, or
- use the registrar's **ROI dropdown + ASSIGN dropdown ROI** button, which does not depend on image clicks.


In [ ]:

_click_test_status = widgets.HTML()
display(_click_test_status)

if CLICK_BACKEND:
    _click_test_fig, _click_test_ax = plt.subplots(figsize=(5.5, 2.2))
    _click_test_ax.imshow(np.linspace(0, 1, 300).reshape(10, 30), cmap="gray", aspect="auto")
    _click_test_ax.set_title("CLICK ANYWHERE IN THIS IMAGE")
    _click_test_ax.set_xticks([]); _click_test_ax.set_yticks([])
    _click_test_status.value = "<b style='color:#b36b00'>Waiting for a test click…</b>"

    def _click_test(event):
        if event.inaxes is _click_test_ax and event.xdata is not None:
            _click_test_status.value = (
                f"<b style='color:#14833b'>CLICK RECEIVED ✓</b> "
                f"x={event.xdata:.1f}, y={event.ydata:.1f}. Direct ROI clicking should work."
            )
            _click_test_ax.plot(event.xdata, event.ydata, "+", ms=18, mew=3)
            _click_test_fig.canvas.draw_idle()

    _click_test_fig.canvas.mpl_connect("button_press_event", _click_test)
    plt.show()
else:
    _click_test_status.value = (
        "<b style='color:#b00020'>DIRECT CLICKING IS DISABLED.</b> "
        "Use the dropdown assignment controls or install <code>ipympl</code> and restart the kernel."
    )


## 1. Choose a mouse

Normally only `SUBJECT_ID` needs to change.

In [ ]:
BASE_PATH = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics")
SUBJECT_ID = 852835

PARADIGMS = ["change_detection_passive"]
EXCLUDE_SESSION_TYPES = ["expression_check", "volume_imaging"]
SELECTED_SESSION_IDS = None   # None = every eligible session for this mouse
TRACE_VARIANT = "dff_robust_f0_trial"

REGISTRATION_FILENAME = "roi_identity_registration.csv"
WIDE_FILENAME = "roi_identity_registration_wide.csv"
BACKUP_DIRNAME = "roi_identity_registration_backups"

# Toggle only the GUI / interaction layer. The saved CSV always contains all anatomical ROI rows.
APPLY_ROI_QC_FILTER = False


# Click hit-testing: exact ROI masks are dilated by this many pixels only for interaction.
# This makes thin/small soma outlines easy to click without changing the saved ROI itself.
CLICK_DILATION_PX = 7

# Crop radius for the active-cell history strip.
HISTORY_CROP_RADIUS_PX = 75

SUPERFICIAL_COLOR = "#eaa186"
DEEP_COLOR = "#4379bc"
SELECTED_COLOR = "#ffd84d"
QC_FAIL_COLOR = "#d62728"
MANUAL_EXCLUDED_COLOR = "#7f7f7f"
UNASSIGNED_COLOR = "#ffffff"


## 2. Load sessions, reference images, ROI masks, and the existing identity table

In [ ]:
def first_existing(paths):
    for path in paths:
        if path is None:
            continue
        path = Path(path)
        if path.exists():
            return path
    return None


def asset_value(asset, name, default=None):
    value = getattr(asset, name, default)
    return value() if callable(value) else value


def make_session_label(row):
    image_set = row.get("image_set", np.nan)
    day = row.get("image_set_day_index", np.nan)
    if pd.notna(image_set) and pd.notna(day):
        return f"{image_set}{int(day)}"
    return str(row.get("session_type", row["session_id"]))


def row_depth(row, dmd):
    for key in (f"dmd{dmd}_depth", f"dmd{dmd}_depth_um", f"DMD{dmd}_depth", f"DMD{dmd}_depth_um"):
        if key in row and pd.notna(row[key]):
            return float(row[key])
    metadata = row.get("metadata", {}) if isinstance(row.get("metadata", {}), dict) else {}
    for key in (f"dmd{dmd}_depth", f"dmd{dmd}_depth_um"):
        if key in metadata and pd.notna(metadata[key]):
            return float(metadata[key])
    return np.nan


def resolve_summary_and_trace(asset):
    voltage_dir = Path(asset.derived_dir) / "voltage"
    qc_dir = Path(asset.qc_dir) / "voltage"
    qc_json = first_existing([
        qc_dir / f"voltage_extraction_qc_{TRACE_VARIANT}.json",
        *sorted(qc_dir.glob("voltage_extraction_qc_*.json")),
    ])
    summary = None
    if qc_json is not None:
        with qc_json.open("r") as f:
            qc = json.load(f)
        metadata = qc.get("metadata", {}) if isinstance(qc.get("metadata", {}), dict) else {}
        value = qc.get("summary_mat", metadata.get("summary_mat"))
        if value:
            summary = Path(value)
    if summary is None or not summary.exists():
        # Extraction summaries sometimes live outside the derived voltage folder.
        candidates = sorted(Path(asset.session_dir).rglob("dendriticVoltageSummary*.mat"))
        summary = candidates[-1] if candidates else None

    trace = voltage_dir / f"voltage_session_traces_{TRACE_VARIANT}.h5"
    if not trace.exists():
        trace = None
    return summary, trace


def matlab_ref(h5, ref):
    return h5[np.asarray(ref).reshape(-1)[0]]


def _canonical_mask_stack(raw_masks):
    masks = np.asarray(raw_masks, dtype=bool)
    while masks.ndim > 3 and 1 in masks.shape:
        masks = np.squeeze(masks, axis=next(i for i, n in enumerate(masks.shape) if n == 1))
    if masks.ndim == 2:
        masks = masks[None, ...]
    if masks.ndim != 3:
        raise ValueError(f"Expected 2-D/3-D masks, got {masks.shape}")
    roi_axis = int(np.argmin(masks.shape))
    masks = np.moveaxis(masks, roi_axis, 0)
    return np.ascontiguousarray(masks.transpose(0, 2, 1), dtype=bool)


def _canonical_reference_image(raw_image, target_shape):
    image = np.asarray(raw_image, dtype=np.float32)
    while image.ndim > 2 and 1 in image.shape:
        image = np.squeeze(image, axis=next(i for i, n in enumerate(image.shape) if n == 1))
    target_shape = tuple(map(int, target_shape))
    candidates = []
    if image.ndim == 2:
        candidates = [image, image.T]
    else:
        for ax0 in range(image.ndim):
            for ax1 in range(ax0 + 1, image.ndim):
                if (image.shape[ax0], image.shape[ax1]) not in (target_shape, target_shape[::-1]):
                    continue
                moved = np.moveaxis(image, (ax0, ax1), (-2, -1))
                for plane in moved.reshape((-1,) + moved.shape[-2:]):
                    candidates.extend([plane, plane.T])
    candidates = [np.asarray(x, np.float32) for x in candidates if x.shape == target_shape]
    if not candidates:
        raise ValueError(f"Cannot orient reference image {image.shape} to {target_shape}")

    def contrast(x):
        finite = x[np.isfinite(x)]
        if not len(finite):
            return -np.inf
        lo, hi = np.percentile(finite, [2, 99.5])
        return float(hi - lo)
    return max(candidates, key=contrast)


def read_reference_and_masks(summary_path, dmd):
    with h5py.File(summary_path, "r") as h5:
        ref_image = matlab_ref(h5, h5["summary/refIM"][int(dmd)-1, 0])
        ref_masks = matlab_ref(h5, h5["summary/masks"][int(dmd)-1, 0])
        masks = _canonical_mask_stack(ref_masks[()])
        image = _canonical_reference_image(ref_image[()], masks.shape[1:])
    return image, masks


def valid_roi_mask(trace_h5, dmd, n_rois):
    if trace_h5 is None:
        return np.ones(n_rois, dtype=bool)
    with h5py.File(trace_h5, "r") as h5:
        group = h5.get(f"DMD{int(dmd)}")
        if group is None or "valid_rois_mask" not in group:
            return np.ones(n_rois, dtype=bool)
        valid = np.asarray(group["valid_rois_mask"][:], dtype=bool)
    if len(valid) != n_rois:
        warnings.warn(f"DMD{dmd}: valid_rois_mask has {len(valid)} rows but masks have {n_rois}; using all masks")
        return np.ones(n_rois, dtype=bool)
    return valid


registry = VIPSessionRegistry.from_basepath(BASE_PATH)
sessions_df = registry.sessions(
    subject_ids=[SUBJECT_ID],
    paradigms=PARADIGMS,
    exclude_session_types=EXCLUDE_SESSION_TYPES,
).copy()

if SELECTED_SESSION_IDS is not None:
    sessions_df = sessions_df[sessions_df["session_id"].astype(str).isin(SELECTED_SESSION_IDS)]

sessions_df["session_datetime"] = pd.to_datetime(
    sessions_df["session_id"].astype(str).str.extract(r"(\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2})")[0],
    format="%Y-%m-%d_%H-%M-%S", errors="coerce",
)
sessions_df = sessions_df.sort_values("session_datetime").reset_index(drop=True)
sessions_df["session_order"] = np.arange(len(sessions_df))
sessions_df["session_label"] = sessions_df.apply(make_session_label, axis=1)

plane_data = {}
manifest_rows = []
assets = {}
subject_dirs = set()

for _, row in sessions_df.iterrows():
    asset = registry.resolve_assets(row)
    assets[str(asset.session_id)] = asset
    subject_dirs.add(Path(asset.session_dir).parent)
    summary_path, trace_h5 = resolve_summary_and_trace(asset)
    if summary_path is None or not Path(summary_path).exists():
        warnings.warn(f"{asset.session_id}: no dendriticVoltageSummary found; skipping session")
        continue

    for dmd in (1, 2):
        try:
            image, masks = read_reference_and_masks(summary_path, dmd)
        except Exception as exc:
            warnings.warn(f"{asset.session_id} DMD{dmd}: {exc}; skipping plane")
            continue
        valid = valid_roi_mask(trace_h5, dmd, len(masks))
        key = (str(asset.session_id), int(dmd))
        plane_data[key] = {
            "image": image,
            "masks": masks,
            "valid": valid,
            "session_label": row["session_label"],
            "session_order": int(row["session_order"]),
            "depth_um": row_depth(row, dmd),
            "session_dir": Path(asset.session_dir),
        }
        for roi, mask in enumerate(masks):
            yy, xx = np.nonzero(mask)
            manifest_rows.append({
                "subject_id": str(asset.subject_id),
                "session_id": str(asset.session_id),
                "session_label": row["session_label"],
                "session_order": int(row["session_order"]),
                "dmd": int(dmd),
                "roi": int(roi),
                "source_roi_label": f"DMD{dmd}_ROI{roi}",
                "depth_um": row_depth(row, dmd),
                "depth_bin_50um": (f"{int(np.floor(row_depth(row, dmd)/50)*50)}–{int(np.floor(row_depth(row, dmd)/50)*50+50)}"
                                   if np.isfinite(row_depth(row, dmd)) else "unknown"),
                "valid_roi": bool(valid[roi]),
                "centroid_x_px": float(np.mean(xx)) if len(xx) else np.nan,
                "centroid_y_px": float(np.mean(yy)) if len(yy) else np.nan,
                "roi_area_px": int(mask.sum()),
                "session_dir": str(asset.session_dir),
            })

if not manifest_rows:
    raise RuntimeError("No reference-image/ROI-mask planes were loaded.")
if len(subject_dirs) != 1:
    raise RuntimeError(f"Expected one subject directory, found: {sorted(map(str, subject_dirs))}")

SUBJECT_DIR = next(iter(subject_dirs))
REGISTRATION_CSV = SUBJECT_DIR / REGISTRATION_FILENAME
WIDE_CSV = SUBJECT_DIR / WIDE_FILENAME
BACKUP_DIR = SUBJECT_DIR / BACKUP_DIRNAME

registration_df = pd.DataFrame(manifest_rows)
for column, default in [
    ("global_cell_id", ""),
    ("excluded", False),
    ("confidence", "high"),
    ("notes", ""),
    ("updated_at", ""),
]:
    registration_df[column] = default

if REGISTRATION_CSV.exists():
    old = pd.read_csv(REGISTRATION_CSV, dtype={"subject_id": str, "session_id": str, "global_cell_id": str})
    keys = ["session_id", "dmd", "roi"]
    preserve = [c for c in ["global_cell_id", "excluded", "confidence", "notes", "updated_at"] if c in old]
    old = old[keys + preserve].drop_duplicates(keys, keep="last")
    registration_df = registration_df.drop(columns=preserve, errors="ignore").merge(old, on=keys, how="left")
    registration_df["global_cell_id"] = registration_df["global_cell_id"].fillna("").replace("nan", "")
    registration_df["excluded"] = registration_df["excluded"].fillna(False).map(lambda x: x if isinstance(x, (bool, np.bool_)) else str(x).strip().lower() in {"true", "1", "yes", "y"}).astype(bool)
    registration_df["confidence"] = registration_df["confidence"].fillna("high")
    registration_df["notes"] = registration_df["notes"].fillna("")
    registration_df["updated_at"] = registration_df["updated_at"].fillna("")
    print("Reloaded existing annotations:", REGISTRATION_CSV)

print(f"Loaded {len(plane_data)} session/DMD panels and {len(registration_df)} ROI rows")
print("Subject directory:", SUBJECT_DIR)
display(registration_df.head())


In [ ]:

def stable_identity_color(cell_id):
    if not cell_id:
        return UNASSIGNED_COLOR
    digest = hashlib.md5(str(cell_id).encode()).hexdigest()
    idx = int(digest[:8], 16) % 20
    return cm.get_cmap("tab20")(idx)


def _boolish(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    return str(value).strip().lower() in {"true", "1", "yes", "y"}


class LongitudinalCellRegistrar:
    """Human-curated, cell-by-cell longitudinal ROI registrar."""

    def __init__(self, plane_data, table, registration_csv, wide_csv, backup_dir):
        self.plane_data = plane_data
        self.table = table.copy()
        self.registration_csv = Path(registration_csv)
        self.wide_csv = Path(wide_csv)
        self.backup_dir = Path(backup_dir)
        self.undo_stack = []
        self.pending_conflict = None
        self.axes_to_key = {}
        self._backup_made = False

        # Normalize editable columns.
        self.table["global_cell_id"] = self.table["global_cell_id"].fillna("").astype(str).replace("nan", "")
        self.table["excluded"] = self.table["excluded"].map(_boolish).astype(bool)
        self.table["confidence"] = self.table["confidence"].fillna("high").astype(str)
        self.table["notes"] = self.table["notes"].fillna("").astype(str)

        self.sessions = []
        for sid, sub in self.table.groupby("session_id", sort=False):
            row = sub.sort_values("session_order").iloc[0]
            self.sessions.append({
                "session_id": str(sid),
                "label": str(row.session_label),
                "order": int(row.session_order),
            })
        self.sessions = sorted(self.sessions, key=lambda x: x["order"])
        self.session_ids = [x["session_id"] for x in self.sessions]
        self.session_label = {x["session_id"]: x["label"] for x in self.sessions}
        self.session_order = {x["session_id"]: x["order"] for x in self.sessions}

        sess_opts = [(f"{x['label']}  ·  {x['session_id']}", x["session_id"]) for x in self.sessions]
        self.anchor = widgets.Dropdown(options=sess_opts, description="Anchor:", layout=widgets.Layout(width="520px"))
        self.working = widgets.Dropdown(options=sess_opts, description="Working:", layout=widgets.Layout(width="520px"))

        ids = self._existing_ids()
        self.cell = widgets.Combobox(
            options=ids, value=(ids[0] if ids else ""), ensure_option=False,
            description="ACTIVE CELL:", layout=widgets.Layout(width="360px")
        )
        self.new_cell_btn = widgets.Button(description="NEW CELL", button_style="success", icon="plus")
        self.prev_btn = widgets.Button(description="PREVIOUS SESSION", icon="arrow-left")
        self.next_btn = widgets.Button(description="NEXT SESSION", icon="arrow-right")
        self.next_missing_btn = widgets.Button(description="NEXT MISSING", button_style="info")
        self.use_anchor_btn = widgets.Button(description="USE WORKING AS ANCHOR")

        self.roi_dropdown = widgets.Dropdown(description="Dropdown ROI:", layout=widgets.Layout(width="330px"))
        self.assign_dropdown_btn = widgets.Button(description="ASSIGN DROPDOWN ROI", button_style="success")
        self.force_btn = widgets.Button(description="FORCE REASSIGN PENDING ROI", button_style="danger", disabled=True)
        self.unassign_session_btn = widgets.Button(description="REMOVE ACTIVE CELL FROM WORKING SESSION", button_style="warning")
        self.undo_btn = widgets.Button(description="UNDO LAST EDIT")
        self.save_btn = widgets.Button(description="SAVE NOW", button_style="primary")

        self.confidence = widgets.Dropdown(options=["high", "medium", "low"], value="high", description="Confidence:")
        self.notes = widgets.Text(description="Notes:", placeholder="optional note for assignments made now", layout=widgets.Layout(width="600px"))

        self.backend_banner = widgets.HTML()
        self.cell_summary = widgets.HTML()
        self.status = widgets.HTML()
        self.legend = widgets.HTML(
            "<b>Viewer:</b> yellow = active cell; white = unassigned; colored = another assigned cell; "
            "red dashed = QC fail; gray dotted = manually excluded. ROI labels are the extraction ROI indices."
        )

        # Wire widgets.
        self.anchor.observe(self._session_changed, names="value")
        self.working.observe(self._session_changed, names="value")
        self.cell.observe(self._cell_changed, names="value")
        self.new_cell_btn.on_click(self._new_cell)
        self.prev_btn.on_click(lambda _: self._step_working(-1))
        self.next_btn.on_click(lambda _: self._step_working(+1))
        self.next_missing_btn.on_click(self._next_missing)
        self.use_anchor_btn.on_click(lambda _: setattr(self.anchor, "value", self.working.value))
        self.assign_dropdown_btn.on_click(self._assign_dropdown)
        self.force_btn.on_click(self._force_pending)
        self.unassign_session_btn.on_click(self._unassign_active_from_working)
        self.undo_btn.on_click(self._undo)
        self.save_btn.on_click(lambda _: self._autosave(message="Saved explicitly."))

        # Establish a clear default: first session as anchor and working.
        if self.session_ids:
            self.anchor.value = self.session_ids[0]
            self.working.value = self.session_ids[0]

        self._make_backup_once()
        self._build_controls()
        self._build_figure()
        self._refresh_roi_dropdown()
        self._refresh_all()

    # ---------- persistence ----------
    def _make_backup_once(self):
        if self._backup_made:
            return
        self.backup_dir.mkdir(parents=True, exist_ok=True)
        if self.registration_csv.exists():
            stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            dst = self.backup_dir / f"roi_identity_registration_before_v2_{stamp}.csv"
            shutil.copy2(self.registration_csv, dst)
        self._backup_made = True

    def _wide_table(self, out):
        source = out[
            out["global_cell_id"].fillna("").astype(str).ne("")
            & ~out["excluded"].astype(bool)
        ].copy()
        if source.empty:
            return pd.DataFrame(columns=["global_cell_id"])
        source["roi_location"] = ("DMD" + source["dmd"].astype(str) + "_ROI" + source["roi"].astype(str) + np.where(source["valid_roi"].astype(bool), "", " [QC_FAIL]"))
        wide = source.pivot_table(
            index="global_cell_id", columns="session_label", values="roi_location",
            aggfunc="first", fill_value=""
        )
        ordered_labels = [self.session_label[s] for s in self.session_ids]
        wide = wide.reindex(columns=[x for x in ordered_labels if x in wide.columns])
        return wide.reset_index()

    def _validate_table(self):
        assigned = self.table[
            self.table["global_cell_id"].fillna("").astype(str).ne("")
            & ~self.table["excluded"].astype(bool)
        ].copy()
        dup = assigned.duplicated(["session_id", "global_cell_id"], keep=False)
        return assigned.loc[dup].sort_values(["session_id", "global_cell_id"])

    def _autosave(self, message=None):
        duplicate = self._validate_table()
        if len(duplicate):
            self._set_status("SAVE BLOCKED: duplicate cell identity exists within a session.", error=True)
            display(duplicate)
            return False
        self.registration_csv.parent.mkdir(parents=True, exist_ok=True)
        out = self.table.sort_values(["session_order", "dmd", "roi"]).reset_index(drop=True).copy()
        tmp = self.registration_csv.with_suffix(self.registration_csv.suffix + ".tmp")
        out.to_csv(tmp, index=False)
        tmp.replace(self.registration_csv)
        wide = self._wide_table(out)
        wide_tmp = self.wide_csv.with_suffix(self.wide_csv.suffix + ".tmp")
        wide.to_csv(wide_tmp, index=False)
        wide_tmp.replace(self.wide_csv)
        if message:
            self._set_status(f"{message}  Autosaved → {self.registration_csv}")
        return True

    # ---------- table helpers ----------
    def _existing_ids(self):
        return sorted(x for x in self.table["global_cell_id"].fillna("").astype(str).unique() if x and x != "nan")

    def _row_mask(self, node):
        sid, dmd, roi = node
        return (
            self.table["session_id"].astype(str).eq(str(sid))
            & self.table["dmd"].eq(int(dmd))
            & self.table["roi"].eq(int(roi))
        )

    def _node_row(self, node):
        m = self._row_mask(node)
        return self.table.loc[m].iloc[0] if m.any() else None

    def _active_cell(self):
        return self.cell.value.strip()

    def _active_assignment(self, session_id):
        cid = self._active_cell()
        if not cid:
            return None
        sub = self.table[
            self.table["session_id"].astype(str).eq(str(session_id))
            & self.table["global_cell_id"].astype(str).eq(cid)
            & ~self.table["excluded"].astype(bool)
        ]
        if sub.empty:
            return None
        r = sub.iloc[0]
        return (str(r.session_id), int(r.dmd), int(r.roi))

    def _snapshot(self):
        return self.table[["session_id", "dmd", "roi", "global_cell_id", "excluded", "confidence", "notes", "updated_at"]].copy(deep=True)

    def _restore_snapshot(self, snap):
        key = ["session_id", "dmd", "roi"]
        keep = ["global_cell_id", "excluded", "confidence", "notes", "updated_at"]
        base = self.table.drop(columns=keep)
        self.table = base.merge(snap[key + keep], on=key, how="left")

    # ---------- ROI visibility ----------
    def _display_roi_indices(self, key):
        """
        Return the ROI indices that should be visible/selectable in the GUI.

        APPLY_ROI_QC_FILTER = False:
            Show every anatomical ROI mask, regardless of automated QC.

        APPLY_ROI_QC_FILTER = True:
            Show only ROIs passing the trace-H5 valid_rois_mask.

        This affects only GUI display/selection. All ROI rows remain present
        in roi_identity_registration.csv.
        """
        plane = self.plane_data[key]

        if APPLY_ROI_QC_FILTER:
            valid = np.asarray(plane["valid"], dtype=bool)

            if len(valid) != len(plane["masks"]):
                warnings.warn(
                    f"{key}: QC mask has {len(valid)} entries but "
                    f"{len(plane['masks'])} anatomical masks exist; "
                    "showing all ROIs instead."
                )
                return np.arange(len(plane["masks"]), dtype=int)

            return np.where(valid)[0].astype(int)

        return np.arange(len(plane["masks"]), dtype=int)

    # ---------- controls ----------
    def _build_controls(self):
        mode_txt = "QC filter ON: only QC-passing ROIs are shown/selectable." if APPLY_ROI_QC_FILTER else "QC filter OFF: all ROIs are shown/selectable." 
        self.backend_banner.value = (
            f"<div style='padding:8px;border-radius:5px;background:#e8f5e9'><b>Direct image clicking: ENABLED ✓</b> — clicking any outlined ROI assigns it to the active cell and autosaves. {mode_txt}</div>"
            if CLICK_BACKEND else
            f"<div style='padding:8px;border-radius:5px;background:#fff3cd'><b>Direct image clicking: DISABLED.</b> Use Dropdown ROI + ASSIGN DROPDOWN ROI. {mode_txt} To enable clicking, install ipympl and restart the kernel.</div>"
        )
        title = widgets.HTML("<h3 style='margin:4px 0'>Longitudinal cell registrar</h3>")
        row_cell = widgets.HBox([self.cell, self.new_cell_btn, self.confidence, self.notes])
        row_sessions = widgets.HBox([self.anchor, self.working])
        row_nav = widgets.HBox([self.prev_btn, self.next_btn, self.next_missing_btn, self.use_anchor_btn])
        row_fallback = widgets.HBox([self.roi_dropdown, self.assign_dropdown_btn, self.force_btn])
        row_edit = widgets.HBox([self.unassign_session_btn, self.undo_btn, self.save_btn])
        display(widgets.VBox([
            title, self.backend_banner, row_cell, row_sessions, row_nav,
            self.legend, row_fallback, row_edit, self.cell_summary, self.status
        ]))

    def _build_figure(self):
        self.fig, self.axes = plt.subplots(2, 2, figsize=(13.5, 9.0), squeeze=False)
        self.fig.subplots_adjust(hspace=0.20, wspace=0.05)
        self.cid_click = None
        if CLICK_BACKEND:
            self.cid_click = self.fig.canvas.mpl_connect("button_press_event", self._onclick)
        plt.show()

    def _session_changed(self, change):
        self.pending_conflict = None
        self.force_btn.disabled = True
        self._refresh_roi_dropdown()
        self._refresh_all()

    def _cell_changed(self, change):
        self.pending_conflict = None
        self.force_btn.disabled = True
        self._refresh_all()

    def _new_cell(self, _):
        existing = self.table["global_cell_id"].fillna("").astype(str)
        pattern = re.compile(rf"^{re.escape(str(SUBJECT_ID))}_C(\d+)$")
        used = [int(m.group(1)) for x in existing for m in [pattern.match(x)] if m]
        next_id = max(used, default=-1) + 1
        cid = f"{SUBJECT_ID}_C{next_id:03d}"
        self.cell.options = sorted(set(self._existing_ids() + [cid]))
        self.cell.value = cid
        self._set_status(f"Created active identity {cid}. Click this biological cell in the anchor session, then advance through days.")

    def _step_working(self, delta):
        if not self.session_ids:
            return
        i = self.session_ids.index(self.working.value)
        j = max(0, min(len(self.session_ids)-1, i + int(delta)))
        self.working.value = self.session_ids[j]

    def _next_missing(self, _):
        if not self._active_cell():
            self._set_status("Create or choose an ACTIVE CELL first.", error=True)
            return
        start = self.session_ids.index(self.working.value)
        candidates = self.session_ids[start+1:] + self.session_ids[:start+1]
        for sid in candidates:
            if self._active_assignment(sid) is None:
                self.working.value = sid
                return
        self._set_status(f"{self._active_cell()} is already assigned in every loaded session.")

    def _refresh_roi_dropdown(self):
        sid = self.working.value
        options = []
        if sid is not None:
            for dmd in (1, 2):
                key = (str(sid), dmd)
                if key not in self.plane_data:
                    continue
                for roi in self._display_roi_indices(key):
                    roi = int(roi)
                    row = self._node_row((sid, dmd, roi))
                    cid = "" if row is None else str(row.global_cell_id or "")
                    qc_text = "QC PASS" if (row is not None and bool(row.valid_roi)) else "QC FAIL"
                    suffix = f"  [{cid}]" if cid else ""
                    options.append((f"DMD{dmd} ROI{roi} · {qc_text}{suffix}", (str(sid), dmd, roi)))
        self.roi_dropdown.options = options
        if options:
            self.roi_dropdown.value = options[0][1]

    # ---------- hit testing and assignment ----------
    def _hit_test(self, key, x, y):
        plane = self.plane_data[key]
        xi, yi = int(round(x)), int(round(y))
        h, w = plane["masks"].shape[1:]
        if not (0 <= xi < w and 0 <= yi < h):
            return None
        roi_idx = self._display_roi_indices(key)
        if not len(roi_idx):
            return None

        # First accept a click actually inside any extracted ROI. Voltage QC does
        # not control anatomical identity annotation.
        inside = [int(r) for r in roi_idx if bool(plane["masks"][int(r), yi, xi])]
        if inside:
            if len(inside) == 1:
                roi = inside[0]
            else:
                roi = min(inside, key=lambda r: self._centroid_distance((key[0], key[1], r), x, y))
            return (str(key[0]), int(key[1]), int(roi))

        # Then allow a small geometric tolerance around the mask boundary.
        try:
            from scipy.ndimage import binary_dilation
            candidates = []
            for r in roi_idx:
                dilated = binary_dilation(plane["masks"][int(r)], iterations=int(CLICK_DILATION_PX))
                if dilated[yi, xi]:
                    candidates.append(int(r))
            if candidates:
                roi = min(candidates, key=lambda r: self._centroid_distance((key[0], key[1], r), x, y))
                return (str(key[0]), int(key[1]), int(roi))
        except Exception:
            pass
        return None

    def _centroid_distance(self, node, x, y):
        row = self._node_row(node)
        if row is None:
            return np.inf
        return float(np.hypot(float(row.centroid_x_px)-x, float(row.centroid_y_px)-y))

    def _onclick(self, event):
        if event.inaxes not in self.axes_to_key or event.xdata is None or event.ydata is None:
            return
        key = self.axes_to_key[event.inaxes]
        node = self._hit_test(key, event.xdata, event.ydata)
        if node is None:
            self._set_status(
                f"Click received in {self.session_label.get(str(key[0]), key[0])} DMD{key[1]}, but it was not on an extracted ROI mask. "
                "Click inside an outlined soma or use Dropdown ROI.",
                error=True,
            )
            return
        self._attempt_assign(node, source="click")

    def _assign_dropdown(self, _):
        if self.roi_dropdown.value is None:
            self._set_status("No dropdown ROI is available in the working session.", error=True)
            return
        self._attempt_assign(tuple(self.roi_dropdown.value), source="dropdown")

    def _attempt_assign(self, node, source="click", force=False):
        cid = self._active_cell()
        if not cid:
            self._set_status("Create or choose an ACTIVE CELL before assigning an ROI.", error=True)
            return
        row = self._node_row(node)
        if row is None:
            self._set_status(f"Could not locate table row for {node}.", error=True)
            return
        if bool(row.excluded):
            self._set_status(f"{self._format_node(node)} is excluded. Clear exclusion in the CSV if this ROI should be used.", error=True)
            return

        target_cid = str(row.global_cell_id or "")
        existing = self._active_assignment(node[0])

        conflict_reasons = []
        if target_cid and target_cid != cid:
            conflict_reasons.append(f"target ROI already belongs to {target_cid}")
        if existing is not None and existing != node:
            conflict_reasons.append(f"{cid} is already assigned to {self._format_node(existing)} in this session")

        if conflict_reasons and not force:
            self.pending_conflict = node
            self.force_btn.disabled = False
            self._set_status(
                "CONFLICT — no change made. " + "; ".join(conflict_reasons) +
                f". Pending target: {self._format_node(node)}. If this is an intentional correction, press FORCE REASSIGN PENDING ROI.",
                error=True,
            )
            self._draw()
            return

        self.undo_stack.append(self._snapshot())
        stamp = datetime.now().isoformat(timespec="seconds")

        # In force mode, free the target from another cell and free the active cell's prior ROI in this session.
        if force:
            if target_cid and target_cid != cid:
                m = self._row_mask(node)
                self.table.loc[m, "global_cell_id"] = ""
            if existing is not None and existing != node:
                m = self._row_mask(existing)
                self.table.loc[m, "global_cell_id"] = ""
                self.table.loc[m, "updated_at"] = stamp

        m = self._row_mask(node)
        self.table.loc[m, "global_cell_id"] = cid
        self.table.loc[m, "excluded"] = False
        self.table.loc[m, "confidence"] = self.confidence.value
        self.table.loc[m, "notes"] = self.notes.value
        self.table.loc[m, "updated_at"] = stamp

        self.pending_conflict = None
        self.force_btn.disabled = True
        self.cell.options = self._existing_ids()
        self._autosave()
        self._refresh_roi_dropdown()
        self._refresh_all()
        qc_note = "" if bool(row.valid_roi) else "  ·  QC FAIL (identity saved; downstream physiology will exclude this observation)"
        self._set_status(
            f"ASSIGNED ✓  {cid} ← {self._format_node(node)} via {source}.{qc_note}  AUTOSAVED → {self.registration_csv}"
        )

    def _force_pending(self, _):
        if self.pending_conflict is None:
            self._set_status("There is no pending conflict to force.", error=True)
            return
        node = self.pending_conflict
        self._attempt_assign(node, source="forced correction", force=True)

    def _unassign_active_from_working(self, _):
        cid = self._active_cell()
        sid = self.working.value
        node = self._active_assignment(sid)
        if not cid or node is None:
            self._set_status("The active cell has no assignment in the working session.", error=True)
            return
        self.undo_stack.append(self._snapshot())
        m = self._row_mask(node)
        self.table.loc[m, "global_cell_id"] = ""
        self.table.loc[m, "updated_at"] = datetime.now().isoformat(timespec="seconds")
        self._autosave()
        self._refresh_roi_dropdown(); self._refresh_all()
        self._set_status(f"Removed {cid} from {self._format_node(node)}. AUTOSAVED.")

    def _undo(self, _):
        if not self.undo_stack:
            self._set_status("Nothing to undo in this run.", error=True)
            return
        snap = self.undo_stack.pop()
        self._restore_snapshot(snap)
        self.pending_conflict = None
        self.force_btn.disabled = True
        self.cell.options = self._existing_ids()
        self._autosave()
        self._refresh_roi_dropdown(); self._refresh_all()
        self._set_status("Undid the last identity edit. AUTOSAVED.")

    # ---------- visual state ----------
    def _format_node(self, node):
        sid, dmd, roi = node
        return f"{self.session_label.get(str(sid), sid)} / DMD{int(dmd)} / ROI{int(roi)}"

    def _set_status(self, text, error=False):
        bg = "#fdecea" if error else "#eaf6ee"
        fg = "#9b1c1c" if error else "#145a32"
        self.status.value = f"<div style='padding:9px;border-radius:5px;background:{bg};color:{fg}'><b>{text}</b></div>"

    def _refresh_summary(self):
        cid = self._active_cell()
        if not cid:
            self.cell_summary.value = "<b>No active cell.</b> Press NEW CELL or choose an existing ID."
            return
        sub = self.table[
            self.table["global_cell_id"].astype(str).eq(cid)
            & ~self.table["excluded"].astype(bool)
        ].sort_values("session_order")
        if sub.empty:
            self.cell_summary.value = f"<b>{cid}</b>: 0 sessions assigned yet."
            return
        pieces = []
        for r in sub.itertuples():
            qc_note = (
                ' <span style="color:#d62728"><b>[QC FAIL]</b></span>'
                if not bool(r.valid_roi)
                else ""
            )
            pieces.append(
                f"<b>{r.session_label}</b>: DMD{int(r.dmd)} R{int(r.roi)} "
                f"({float(r.depth_um):.0f} µm){qc_note}"
            )
        self.cell_summary.value = (
            f"<div style='padding:7px;border:1px solid #ddd'><b>{cid}</b> — {sub['session_id'].nunique()} session(s): "
            + " &nbsp; | &nbsp; ".join(pieces) + "</div>"
        )

    def _draw_panel(self, ax, sid, dmd, row_name):
        ax.clear()
        key = (str(sid), int(dmd))
        if key not in self.plane_data:
            ax.axis("off")
            ax.set_title(f"{row_name}: {self.session_label.get(str(sid), sid)} · DMD{dmd} · unavailable")
            return
        self.axes_to_key[ax] = key
        plane = self.plane_data[key]
        image = plane["image"]
        finite = image[np.isfinite(image)]
        lo, hi = np.percentile(finite, [2, 99.5]) if len(finite) else (0, 1)
        if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
            lo, hi = np.nanmin(image), np.nanmax(image)
        ax.imshow(image, cmap="gray", vmin=lo, vmax=hi)

        rows = self.table[
            self.table["session_id"].astype(str).eq(str(sid))
            & self.table["dmd"].eq(int(dmd))
        ].set_index("roi")
        active = self._active_cell()
        pending = self.pending_conflict

        for roi in self._display_roi_indices(key):
            roi = int(roi)
            if roi not in rows.index:
                continue
            mask = plane["masks"][roi]
            r = rows.loc[roi]
            node = (str(sid), int(dmd), int(roi))
            cid = str(r.global_cell_id) if pd.notna(r.global_cell_id) and str(r.global_cell_id) != "nan" else ""
            qc_valid = bool(r.valid_roi)

            if bool(r.excluded):
                color, lw, z, linestyle = MANUAL_EXCLUDED_COLOR, 2.0, 4, ":"
            elif cid and cid == active:
                color, lw, z, linestyle = SELECTED_COLOR, 3.5, 8, ("-" if qc_valid else "--")
            elif pending is not None and node == pending:
                color, lw, z, linestyle = "#ff00ff", 3.5, 9, ("-" if qc_valid else "--")
            elif cid:
                color, lw, z, linestyle = stable_identity_color(cid), 2.0, 5, ("-" if qc_valid else "--")
            else:
                color = UNASSIGNED_COLOR if qc_valid else QC_FAIL_COLOR
                lw, z, linestyle = (1.5 if qc_valid else 2.2), 3, ("-" if qc_valid else "--")

            ax.contour(mask, levels=[0.5], colors=[color], linewidths=lw, linestyles=linestyle, zorder=z)
            x, y = float(r.centroid_x_px), float(r.centroid_y_px)
            suffix = cid.replace(f"{SUBJECT_ID}_", "") if cid else ""
            qc_suffix = "\nQC FAIL" if not qc_valid else ""
            text = f"R{roi}" + (f"\n{suffix}" if suffix else "") + qc_suffix
            bbox_edge = QC_FAIL_COLOR if not qc_valid else color
            ax.text(
                x, y, text, ha="center", va="center", fontsize=8, color="black", zorder=10,
                bbox={"facecolor":"white", "edgecolor":bbox_edge, "alpha":0.86, "pad":1.1}
            )
        depth = plane["depth_um"]
        depth_txt = f"{depth:.0f} µm" if np.isfinite(depth) else "depth unknown"
        ax.set_title(f"{row_name}: {self.session_label.get(str(sid), sid)} · DMD{dmd} · {depth_txt}")
        ax.set_axis_off()

    def _draw(self):
        self.axes_to_key = {}
        anchor_sid = self.anchor.value
        working_sid = self.working.value
        self._draw_panel(self.axes[0,0], anchor_sid, 1, "ANCHOR")
        self._draw_panel(self.axes[0,1], anchor_sid, 2, "ANCHOR")
        self._draw_panel(self.axes[1,0], working_sid, 1, "WORKING")
        self._draw_panel(self.axes[1,1], working_sid, 2, "WORKING")
        active = self._active_cell() or "<none>"
        qc_mode = "QC filter ON" if APPLY_ROI_QC_FILTER else "QC filter OFF"
        self.fig.suptitle(
            f"Subject {SUBJECT_ID} · ACTIVE CELL {active} · {qc_mode} · click an outlined ROI to assign",
            fontsize=14, fontweight="bold"
        )
        self.fig.canvas.draw_idle()

    def _refresh_all(self):
        self._refresh_summary()
        self._draw()

    # ---------- public QC helpers ----------
    def current_table(self):
        return self.table.sort_values(["session_order", "dmd", "roi"]).reset_index(drop=True).copy()

    def assigned_table(self):
        return self.current_table()[lambda x: x["global_cell_id"].fillna("").astype(str).ne("")].copy()

    def occupancy_table(self):
        return self._wide_table(self.current_table())


registrar = LongitudinalCellRegistrar(
    plane_data=plane_data,
    table=registration_df,
    registration_csv=REGISTRATION_CSV,
    wide_csv=WIDE_CSV,
    backup_dir=BACKUP_DIR,
)


## 3. Launch the registrar

### Recommended workflow

- Leave `APPLY_ROI_QC_FILTER = False` if you want to **see and annotate every extracted ROI**.
- Click **NEW CELL**. This creates, for example, `852835_C000` and makes it the active biological cell.
- Set the **Anchor** session to a day where this cell is especially obvious.
- Set **Working** to the same session initially and click the cell once to register the anchor observation.
- Advance with **NEXT SESSION**. Compare the anchor row against the working row, then click the matching ROI in either DMD.
- Continue through all days. Use **NEXT MISSING** to skip directly to the next session where the active cell has not yet been assigned.
- When that biological cell is complete, press **NEW CELL** and repeat.

### What the QC toggle does

`APPLY_ROI_QC_FILTER` affects only:
- which ROIs are **drawn**,
- which ROIs appear in **Dropdown ROI**, and
- which ROIs can be selected by **clicking**.

It does **not** change the saved row inventory of `roi_identity_registration.csv`. The CSV always contains all anatomical ROIs loaded from the extraction summaries.

### What a click does

A click is accepted only if it lands **inside or very close to a displayed ROI mask**. The hit target is a small dilation around the actual mask—not an arbitrary nearest-centroid radius. On success, the status box explicitly says which `session / DMD / ROI` was assigned and confirms autosave.

### If clicking does not work

Use **Dropdown ROI** to choose `DMD1 ROI…` or `DMD2 ROI…`, then press **ASSIGN DROPDOWN ROI**. This performs exactly the same assignment and autosave without relying on matplotlib events.

### Conflict behavior

If the target ROI already belongs to another cell, or the active cell already has a different ROI in that session, the assignment is **not changed**. The proposed change appears as a pending conflict. Inspect it, then press **FORCE REASSIGN PENDING ROI** only if the correction is intentional.


## 4. QC the completed registration

After you finish a mouse, run the next cell.

Important distinction:

- `roi_identity_registration.csv` is the **full ROI manifest** and should contain every anatomical ROI row for this mouse.
- `global_cell_id` is blank for ROIs you have not assigned yet.
- `valid_roi` records automated QC status, but the row remains present regardless of the GUI QC toggle.
- The wide occupancy table summarizes only **assigned** biological identities.

Before moving on to the ephys/DoC notebooks, visually inspect any cell that unexpectedly disappears and reappears, switches DMD, or has a low-confidence note.


In [ ]:

assigned = registrar.assigned_table()
wide = registrar.occupancy_table()

display(wide)
current = registrar.current_table()
print(f"Total ROI rows in full registration table: {len(current)}")
print(f"Assigned ROI observations: {len(assigned)}")
print(f"Biological cell IDs: {assigned['global_cell_id'].nunique() if len(assigned) else 0}")
n_qc_fail_all = int((~current["valid_roi"].astype(bool)).sum())
n_qc_fail_assigned = int((~assigned["valid_roi"].astype(bool)).sum()) if len(assigned) else 0
print(f"APPLY_ROI_QC_FILTER = {APPLY_ROI_QC_FILTER}")
print(f"QC-failing ROI rows present in full registration table: {n_qc_fail_all}")
print(f"QC-failing observations with longitudinal identities: {n_qc_fail_assigned}")
if n_qc_fail_assigned:
    print("These identities are retained in the CSV. Downstream notebooks decide separately whether automated QC should exclude them.")
print("Canonical file:", REGISTRATION_CSV)
print("Wide QC file:", WIDE_CSV)

# Hard validation: one biological cell can appear only once per session.
duplicate = registrar._validate_table()
if len(duplicate):
    display(duplicate)
    raise RuntimeError("Registration contains duplicate within-session cell identities; correct these before downstream analysis.")
else:
    print("Validation passed: no global_cell_id occurs more than once within a session.")



## 5. Emergency/manual CSV editing

The GUI writes a simple long-form CSV. If you ever need to repair an identity outside the widget interface, edit only these columns for the relevant `(session_id, dmd, roi)` row:

- `global_cell_id`
- `excluded`
- `confidence`
- `notes`

Then restart the kernel and rerun the notebook so the table is reloaded cleanly. Never assign the same `global_cell_id` to two ROIs in the same session.
